In [3]:
# evaluate the mask rcnn model on the kangaroo dataset
from os import listdir
from xml.etree import ElementTree
from numpy import zeros
from numpy import asarray
from numpy import expand_dims
from numpy import mean
from mrcnn.config import Config
from mrcnn.model import MaskRCNN
from mrcnn.utils import Dataset
from mrcnn.utils import compute_ap
from mrcnn.model import load_image_gt
from mrcnn.model import mold_image
import os
import json
from PIL import Image, ImageDraw
import numpy as np

from config_onion_20201022 import *
from dataset_config import *

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "4"


# define the prediction configuration
class PredictionConfig(Config):
    # define the name of the configuration
    NAME = "onioncell"
    # number of classes (background + kangaroo)
    NUM_CLASSES = 1 + 5
    # simplify GPU config
    GPU_COUNT = 1
    IMAGES_PER_GPU = 1
    
    IMAGE_RESIZE_MODE = "crop"
    IMAGE_MIN_DIM = 512
    IMAGE_MAX_DIM = 512

# calculate the mAP for a model on a given dataset
def evaluate_model(dataset, model, cfg):
    APs = list()
    for image_id in dataset.image_ids:
        # load image, bounding boxes and masks for the image id
        image, image_meta, gt_class_id, gt_bbox, gt_mask = load_image_gt(dataset, cfg, image_id, use_mini_mask=False)
        # convert pixel values (e.g. center)
        scaled_image = mold_image(image, cfg)
        # convert image into one sample
        sample = expand_dims(scaled_image, 0)
        # make prediction
        yhat = model.detect(sample, verbose=0)
        # extract results for first sample
        r = yhat[0]
        # calculate statistics, including AP
        AP, _, _, _ = compute_ap(gt_bbox, gt_class_id, gt_mask, r["rois"], r["class_ids"], r["scores"], r['masks'])
        # store
        APs.append(AP)
    # calculate the mean AP across all images
    mAP = mean(APs)
    return mAP

# load the train dataset
dataset_train = CustomDataset()
dataset_train.load_data('./datasets/COCO/train/annotations/instances_default.json', './datasets/COCO/train/images')
dataset_train.prepare()

# load the test dataset
dataset_val = CustomDataset()
dataset_val.load_data('./datasets/COCO/val/annotations/instances_default.json', './datasets/COCO/val/images')
dataset_val.prepare()

# create config
cfg = PredictionConfig()

# define the model
model = MaskRCNN(mode='inference', model_dir='./', config=cfg)

# load model weights
model_path = 'onioncell_102020201022T0446/mask_rcnn_onioncell_1020_0012.h5'

model.load_weights(model_path, by_name=True)


Instructions for updating:
Colocations handled automatically by placer.
Instructions for updating:
Use tf.cast instead.


In [4]:
# evaluate model on training dataset
train_mAP = evaluate_model(dataset_train, model, cfg)
print("Train mAP: %.3f" % train_mAP)

# evaluate model on test dataset
test_mAP = evaluate_model(dataset_val, model, cfg)
print("Test mAP: %.3f" % test_mAP)

/home/cvat/.virtualenvs/mrcnn/lib/python3.6/site-packages/mask_rcnn-2.1-py3.6.egg/mrcnn/utils.py:734: RuntimeWarning: invalid value encountered in true_divide
/home/cvat/.virtualenvs/mrcnn/lib/python3.6/site-packages/mask_rcnn-2.1-py3.6.egg/mrcnn/utils.py:734: RuntimeWarning: invalid value encountered in true_divide


Train mAP: nan


/home/cvat/.virtualenvs/mrcnn/lib/python3.6/site-packages/mask_rcnn-2.1-py3.6.egg/mrcnn/utils.py:734: RuntimeWarning: invalid value encountered in true_divide


Test mAP: nan
